# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DharshanS2006/flyrank-ml-intern-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

The intended unit of analysis is one daily observation for one pseudonymized content item belonging to one pseudonymized client on a single report date.

Time Window:
The dataset contains historical daily observations across the available reporting period.

Verification:
The grain was checked using the combination (report_date, client_hash_id, content_hash_id). The verification found 78,835,655 total rows and 78,829,265 unique key combinations, indicating 6,390 duplicate key combinations. Therefore, this combination is the intended grain but is not perfectly unique.

In [9]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        CAST(report_date AS VARCHAR) ||
        client_hash_id ||
        content_hash_id
    )) AS unique_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
);
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,78835655,78829265


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Feature

- gsc_impressions
- gsc_sum_position
- sessions_ai
- scroll_events

## Label

- gsc_clicks

## Context

- report_date
- month
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

## Excluded

- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

Reason:
These AI-source columns are not required for this analysis.

Hash identifiers are excluded from modeling because they identify entities but do not provide predictive information.

In [10]:
query = """
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1;
"""

con.sql(query).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification

The following checks verify the data contract:

- Total number of rows
- Duplicate grain check
- Missing values
- Date window

In [11]:
# Total rows

query = """
SELECT COUNT(*) AS total_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
);
"""

print(con.sql(query).df())

# Duplicate grain check

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20;
"""

con.sql(query).df()

# Missing values

query = """
SELECT

SUM(gsc_impressions IS NULL) AS missing_impressions,
SUM(gsc_clicks IS NULL) AS missing_clicks,
SUM(gsc_sum_position IS NULL) AS missing_position,
SUM(report_date IS NULL) AS missing_report_date

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
);
"""

con.sql(query).df()


# Date range

query = """
SELECT

MIN(report_date) AS first_date,
MAX(report_date) AS last_date

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
);
"""

con.sql(query).df()

   total_rows
0    78835655


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count
0,2026-06-16,client_9c26c096d6e57253,content_70b02f5039a882c2,2
1,2026-06-14,client_9c26c096d6e57253,content_01ca8d0758d6614a,2
2,2026-06-14,client_1a8bf67cad4ee525,content_3084acbc2aca184b,2
3,2026-06-19,client_def0955f7a377868,content_c52c3798a1412de7,2
4,2026-06-20,client_9c26c096d6e57253,content_8d81ee3f670fcc71,2
5,2026-06-18,client_b77d0d5f08f05e64,content_6b6a532cb09428b2,2
6,2026-06-17,client_1a8bf67cad4ee525,content_dd441f559f3cb4f7,2
7,2026-06-17,client_1a8bf67cad4ee525,content_10c2a0e85769f485,2
8,2026-06-21,client_810019792c9b8efc,content_41a28b3332556857,2
9,2026-06-21,client_1a730cb2640a1abf,content_62a6c7628cfd006f,2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- The dataset is pseudonymized, so client names, URLs, keywords, and content text are not available.
- Client history is unbalanced because different clients entered the dataset at different times.
- Some observations contain only GSC data or only GA4 data.
- A small number of duplicate key combinations were observed at the intended grain.
- The data supports observational analysis and decision support only. It cannot establish causal relationships.

In [12]:
query = """
SELECT

client_has_gsc,
client_has_ga4,
gsc_data_available,
ga4_data_available,
COUNT(*) AS rows

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)

GROUP BY
client_has_gsc,
client_has_ga4,
gsc_data_available,
ga4_data_available

ORDER BY rows DESC;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,rows
0,True,True,False,False,28912779
1,True,False,False,<NA>,20492434
2,True,True,True,False,15043370
3,True,False,True,<NA>,9142893
4,True,True,True,True,2453375
5,True,False,True,False,2330413
6,True,True,False,True,362385
7,False,True,<NA>,False,97311
8,False,True,<NA>,True,695


## Self-check

Before you submit, confirm each line honestly:

- [ ✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅ ] No client names, URLs, or private queries anywhere
- [ ✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.